# Object-oriented programming in Python

**Class 4**

Before: [gradebook with functions](09-implementation-workshop.ipynb)

After the opening gradebook workshop, we redesign the same problem using objects. By the end, you can define a class, create independent instances, distinguish attributes from local variables, write methods that maintain valid state, and combine objects using composition.

## 1. From records and functions to objects

An object combines **state** (stored data) with **behaviour** (operations). A **class** defines a kind of object; an **instance** is one individual object made from that class.

_Example:_ Car is a class, with some state (e.g., license plate, color, ...) and some functions (e.g., start engine); my_car is an instance of the class Car.

Python strings, lists, and dictionaries are already objects: `scores.append(18)` calls a method on a list. We are now defining our own kind of object.

Objects are useful when related operations share state and rules. Small stateless calculations can remain ordinary functions.

## 2. Define a class and initialise its instances

`__init__` initialises an instance when you create it. The conventional first parameter `self` refers to the instance receiving the method call. `self.value` is stored on that instance; `start` is a parameter of this call.

Calling `first.increment()` supplies `first` as `self` automatically. The method below changes state and also returns the updated value. These are separate design choices.

In [ ]:
class Counter:
    """A counter whose instances keep independent numeric values."""

    def __init__(self, start=0):
        self.value = start

    def increment(self):
        self.value = self.value + 1
        return self.value

first = Counter()
second = Counter(10)
print(first.increment())
print(first.value)
print(second.value)
assert first.value == 1
assert second.value == 10

## Trace identity and state

Two names can refer to the same instance, just as they can refer to the same list. `is` tests identity: whether two names refer to the very same object.

Predict the following outputs. How many Counter instances exist?

In [ ]:
alias = first
alias.increment()
print(first.value)
print(alias is first)
print(second is first)
assert first.value == 2
assert second.value == 10

## Core exercise — Rectangle

Define `Rectangle(width, height)`. Assume positive numeric dimensions for this first version.

- Store `width` and `height` as instance attributes.
- `area()` returns width × height.
- `perimeter()` returns 2 × (width + height).
- `scale(factor)` multiplies both dimensions by a positive factor and returns `None`.

Create rectangles (3, 4) and (2, 5). Their areas are 12 and 10; perimeters are 14 and 14. Scale the first by 2: its dimensions become (6, 8), area 48, perimeter 28. The second must stay unchanged.

Which methods only inspect state, and which modify it? Why should `area` calculate from the dimensions instead of storing an area that may become stale?

In [ ]:
# Define Rectangle and assert its results and independent instance state.

## 3. Maintain valid state

An **invariant** is a rule that should hold after construction and after every public operation. For a student, every recorded score should be between 0 and 20.

`raise ValueError(...)` reports an unacceptable value. A caller can handle it with `try` / `except ValueError`. Validate **before** changing state so a rejected operation leaves the object unchanged.

For the examples here, assume names are nonempty strings and scores are integers. We validate the score range; full type validation is beyond this exercise.

A leading underscore, as in `_scores`, marks an attribute as internal by convention; it is not access protection enforced by Python.

In [ ]:
class Student:
    """A named student with integer scores from 0 to 20."""

    def __init__(self, name):
        self.name = name
        self._scores = []

    def add_score(self, score):
        if score < 0 or score > 20:
            raise ValueError("score must be between 0 and 20")
        self._scores.append(score)

    def mean(self):
        if not self._scores:
            return None
        total = 0
        for score in self._scores:
            total = total + score
        return total / len(self._scores)

    def scores(self):
        return self._scores.copy()

ada = Student("Ada")
sam = Student("Sam")
ada.add_score(12)
ada.add_score(18)
assert ada.mean() == 15.0
assert sam.mean() is None
print(ada.name, ada.mean())

### Worked checks — failure and ownership

A copy returned by `scores()` lets callers inspect the data without changing the internal list through that returned value. The copy is sufficient here because scores are immutable integers.

Trace the failed call: which statements run, and which are skipped? Catch only the error you expect.

In [ ]:
before = ada.scores()
try:
    ada.add_score(21)
except ValueError as error:
    print(error)
else:
    raise AssertionError("An invalid score should have been rejected")
assert ada.scores() == before

snapshot = ada.scores()
snapshot.append(0)
assert ada.scores() == [12, 18]
assert sam.scores() == []

## Core debugging exercise — accidentally shared state

This deliberately faulty code is shown as text. Predict what Ben sees after Ada receives a score:

```python
class BrokenStudent:
    scores = []

    def __init__(self, name):
        self.name = name

ada = BrokenStudent("Ada")
ben = BrokenStudent("Ben")
ada.scores.append(18)
```

The list is a **class attribute**, shared through the class. Fix it by initialising an instance list in `__init__`, then prove that the two students are independent.

**Related trap:** `def __init__(self, scores=[]): ...` creates a default list once when the function is defined. If accepting initial scores, use `scores=None` and make a fresh list inside. Copy a caller-provided list if your object should own its own list. Do not merely store another reference to it.

In [ ]:
# Reproduce the shared-state issue, then write and test a corrected class.

## Core exercise — Extend Student behaviour

Add `status(threshold=10)` to Student. It must return `"ungraded"` for no scores, `"pass"` for mean ≥ threshold, and `"fail"` otherwise. Call `mean()` rather than duplicating its loop. Assume threshold is numeric and from 0 to 20.

Add `__str__(self)` returning a readable description such as `"Ada: 15.0 - pass"` or `"Sam: ungraded"`. Python calls this special method for `str(student)` and `print(student)`. Use the unrounded mean for the status decision and one decimal place for display.

Edit the class definition above and rerun it, then **recreate the instances**. Rerunning a class definition creates a new class; old instances do not automatically gain its new methods.

Test empty, boundary-10, failing, and passing cases. Ada should fail with threshold 16. Test a student with scores [9, 10]: a displayed 9.5 must remain a fail.

In [ ]:
# Recreate students after extending Student; test status and str.

## 4. Composition — an object containing other objects

A group **has** students. This is composition: store references to Student instances inside another object. Methods delegate work to those students, rather than duplicating their mean calculation.

The small example copies the outer list but deliberately shares the Student objects. Adding a score to a student will be reflected in the group’s next report. Replacing or appending items in the caller’s original list will not change membership in this group.

In [ ]:
class StudyGroup:
    def __init__(self, name, students):
        self.name = name
        self._students = list(students)

    def means(self):
        result = {}
        for student in self._students:
            result[student.name] = student.mean()
        return result

lee = Student("Lee")
lee.add_score(10)
members = [lee]
group = StudyGroup("Python", members)
members.append(Student("Jo"))
assert group.means() == {"Lee": 10.0}
lee.add_score(20)
assert group.means() == {"Lee": 15.0}
print(group.means())

## Final project — Gradebook made of objects

Build `Gradebook(title)` using your completed Student class. Assume student names are unique identifiers within a gradebook and must be nonempty strings. Store students in an instance dictionary keyed by name.

Implement:

- `add_student(student)`: store an existing Student object; raise `ValueError` for a duplicate name. Return `None` on success.
- `record_score(name, score)`: find that student and delegate to `add_score`. Raise `KeyError` for an unknown name; return `None` on success. Dictionary lookup naturally raises `KeyError` for missing keys.
- `summary(threshold=10)`: return `{"pass": ..., "fail": ..., "ungraded": ...}` containing student counts. Call each student’s `status`. An empty gradebook returns three zeros.

The gradebook intentionally shares its Student instances with the caller. Treat student names as fixed after registration so dictionary keys stay consistent. Do not change a Student’s internal score list directly.

Before coding, draw the Gradebook object, its dictionary, and three Student objects. Mark which object owns each score list. Contrast this diagram with your function-call diagram from class 3.

In [ ]:
# Define Gradebook, using Student methods for score updates and status.

### Project acceptance scenarios

Create Ada, Sam, and Lee with no scores, and register them in a gradebook. Record 12 and 18 for Ada and 8 for Sam.

| Scenario | Expected result |
| --- | --- |
| Default summary | `{"pass": 1, "fail": 1, "ungraded": 1}` |
| Summary with threshold 16 | `{"pass": 0, "fail": 2, "ungraded": 1}` |
| Empty gradebook | `{"pass": 0, "fail": 0, "ungraded": 0}` |
| Register another student named Ada | ValueError; original Ada retained |
| Record 21 for Ada | ValueError; her scores stay [12, 18] |
| Record a score for an unknown name | KeyError; no student added |
| Add 20 to the original Sam object | Gradebook sees Sam’s new mean 14.0 |

Write assertions and explicit exception checks following the worked example. Create a second Gradebook and check independent membership. Read a summary twice and verify it does not change scores.

**Deliverable:** Student and Gradebook classes, tests, an object diagram, and an explanation of one advantage and one tradeoff compared with the functional version. Both designs should represent the same grading rules.

In [ ]:
# Test the project scenarios and record the functional/OOP comparison.

## Extension bank — deepen the object model

Complete the core project first. Choose one extension and add its own tests.

1. **Rectangle validation:** reject nonpositive dimensions at construction and nonpositive scale factors. Failed scaling must leave both dimensions unchanged.
2. **Score removal:** add `remove_last_score()` returning the removed score, or `None` when empty. Check the mean and status after removing the last score.
3. **Alternative grading:** define a second class `PassFailStudent` with `name` and `status(threshold=10)` returning a stored pass/fail result, ignoring threshold. A Gradebook can summarise it because it provides the required behaviour. It cannot record numeric scores unless you also define that operation. This demonstrates that compatibility depends on the methods used.
4. **Representation:** add `__repr__` showing a useful developer-facing description. Compare a list of objects with printing one object using `__str__`.

## Review — explain the design

- Distinguish a class, an instance, an attribute, and a method using your code.
- Explain why `self` is needed and how it is supplied in a method call.
- Show a query method and a state-changing method.
- Explain where invalid scores are rejected and why rejection happens before mutation.
- Identify one shared object and one independent object in your tests.
- Explain why Gradebook delegates grading behaviour to Student.

You now have two implementations of the same problem. Choose functions or objects according to which makes the data, dependencies, and rules easiest to understand.